In [4]:
# =====================================================
# 02 - Isolation Forest
# Mobile API Misuse Detector
# =====================================================

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest


# =====================================================
# 1. Chemin du projet
# =====================================================

project_path = "/content/drive/MyDrive/Mobile_API_Misuse_Detector/"


# =====================================================
# 2. Charger le dataset final des features
# =====================================================

features_df = pd.read_csv(
    project_path + "data/processed/features_dataset.csv"
)

print("Dataset chargé :")
print(features_df.shape)

features_df.head()



# =====================================================
# 5. Préparer les données ML
# =====================================================

ip_addresses = features_df["ip"]

X = features_df.drop(columns=["ip"])

print("Features utilisées :")
print(X.columns.tolist())


# =====================================================
# 6. Normalisation
# =====================================================

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

joblib.dump(
    scaler,
    project_path + "models/scaler.pkl"
)

print("Scaler sauvegardé.")


# =====================================================
# 7. Entraîner Isolation Forest
# =====================================================

iso_model = IsolationForest(
    n_estimators=100,
    contamination=0.02,
    random_state=42
)

iso_model.fit(X_scaled)

print("Modèle Isolation Forest entraîné.")


# =====================================================
# 8. Prédiction des anomalies
# =====================================================

features_df["iso_prediction"] = iso_model.predict(X_scaled)

# 1 = normal
# -1 = anomalie


# =====================================================
# 9. Score d'anomalie
# =====================================================

features_df["iso_score_raw"] = iso_model.decision_function(X_scaled)

# Plus iso_score_raw est petit, plus l'IP est suspecte.


# =====================================================
# 10. Afficher le nombre normal/anomalie
# =====================================================

print(features_df["iso_prediction"].value_counts())


# =====================================================
# 11. Afficher les IP suspectes
# =====================================================

anomalies = features_df[
    features_df["iso_prediction"] == -1
].sort_values(
    by="iso_score_raw",
    ascending=True
)

anomalies.head(20)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset chargé :
(2173, 20)
Features utilisées :
['request_count', 'max_req_count_5min', 'avg_req_count_5min', 'requests_per_second', 'avg_time_between_requests', 'unique_endpoints', 'unique_ids_accessed', 'status_404_count', 'max_error_rate_5min', 'suspicious_ua_ratio', 'bot_ratio', 'mobile_ratio', 'post_frequency', 'max_repeated_endpoint_hits', 'login_attempt_count', 'failed_login_count', 'failed_login_rate', 'max_login_req_per_min', 'avg_time_between_login_attempts']
Scaler sauvegardé.
Modèle Isolation Forest entraîné.
iso_prediction
 1    2129
-1      44
Name: count, dtype: int64


,ip,request_count,max_req_count_5min,avg_req_count_5min,requests_per_second,avg_time_between_requests,unique_endpoints,unique_ids_accessed,status_404_count,max_error_rate_5min,...,mobile_ratio,post_frequency,max_repeated_endpoint_hits,login_attempt_count,failed_login_count,failed_login_rate,max_login_req_per_min,avg_time_between_login_attempts,iso_prediction,iso_score_raw
1828,66.249.66.194,6557,243,172.552632,0.594793,1.681513,5561,2409,459,0.246988,...,0.589904,0.000000,33,0,0,0.0,0.0,0.000000,-1,-0.214879
561,40.77.167.170,1653,80,50.090909,0.174958,5.719128,1638,959,11,0.030769,...,0.018754,0.000000,4,16,0,0.0,1.0,624.600000,-1,-0.204939
1835,66.249.66.91,4160,202,109.473684,0.377393,2.650397,3789,1273,2,0.056604,...,0.087019,0.000000,349,0,0,0.0,0.0,0.000000,-1,-0.203425
411,207.46.13.9,1276,81,49.076923,0.174103,5.748235,1261,812,14,0.068966,...,0.014890,0.000000,2,6,0,0.0,1.0,1387.600000,-1,-0.199319
558,40.77.167.13,1059,68,52.950000,0.181522,5.514178,1043,717,11,0.057692,...,0.013220,0.000000,2,15,0,0.0,1.0,389.285714,-1,-0.190150
121,157.55.39.245,780,94,37.142857,0.115010,8.706033,763,723,12,0.111111,...,0.030769,0.000000,2,5,0,0.0,1.0,546.000000,-1,-0.189086
408,207.46.13.136,882,65,49.000000,0.170336,5.877412,876,651,10,0.058824,...,0.006803,0.000000,2,8,0,0.0,2.0,526.000000,-1,-0.188554
119,157.55.39.220,842,77,52.625000,0.188072,5.323424,829,646,5,0.020000,...,0.020190,0.000000,2,10,0,0.0,1.0,457.888889,-1,-0.185372
143,17.58.102.43,577,23,15.594595,0.052445,19.100694,561,520,11,0.260870,...,0.000000,0.000000,5,1,0,0.0,1.0,0.000000,-1,-0.164276
2127,91.99.72.15,1725,121,45.394737,0.156463,6.395012,1624,3730,6,0.069767,...,0.000000,0.000000,9,0,0,0.0,0.0,0.000000,-1,-0.150107


In [2]:
# =====================================================
# 12. Sauvegarder modèle + résultats
# =====================================================

joblib.dump(
    iso_model,
    project_path + "models/isolation_forest.pkl"
)

features_df.to_csv(
    project_path + "data/processed/isolation_forest_results.csv",
    index=False
)

print("Modèle et résultats sauvegardés avec succès.")

Modèle et résultats sauvegardés avec succès.
